# Diebold-Li con Kalman y cambio de régimen

Este notebook extiende el flujo base de Diebold-Li + Kalman con un bloque adicional de **cambio de régimen**.

La idea es simple:

1. estimar factores `level`, `slope`, `curvature` con un modelo dinámico tipo Diebold-Li;
2. suavizar esos factores con filtro de Kalman y smoothing RTS;
3. identificar dos regímenes en el espacio de factores;
4. estimar dinámica temporal específica por régimen;
5. comparar curvas y forecasts condicionales al régimen actual.

Importante:
esta versión **no** es un Markov-switching state-space full maximum likelihood. Es una extensión intermedia, seria y útil, construida sobre el flujo Kalman ya implementado en el repo.

## 1. Intuición

El modelo Diebold-Li estándar asume una única dinámica temporal para los factores:

$$
X_t = c + A X_{t-1} + \eta_t
$$

Pero en la práctica la curva puede comportarse distinto en etapas diferentes, por ejemplo:
- regímenes de tasas bajas versus tasas altas;
- episodios de mayor pendiente versus curva más plana;
- períodos de mayor o menor volatilidad.

Por eso, aquí introducimos una variable de régimen `S_t \in \{0,1\}` y permitimos que la dinámica cambie según el estado:

$$
X_t = c_{S_t} + A_{S_t} X_{t-1} + \eta_{t,S_t}
$$

En vez de estimar `S_t` con un Hamilton filter completo, identificamos regímenes sobre los factores suavizados usando una mezcla gaussiana de dos componentes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_lyapunov
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / 'api_js' / 'data' / 'market_rates.csv'

RATE_MONTHS = {
    'TPM': 1,
    'SPC_03Y': 3,
    'SPC_06Y': 6,
    'SPC_1Y': 12,
    'SPC_2Y': 24,
    'SPC_3Y': 36,
    'SPC_4Y': 48,
    'SPC_5Y': 60,
    'SPC_10Y': 120,
}

COLUMNS = list(RATE_MONTHS.keys())
TAU_MONTHS = np.array([RATE_MONTHS[c] for c in COLUMNS], dtype=float)
TAU_YEARS = TAU_MONTHS / 12.0

print(DATA_PATH)

In [ ]:
def load_monthly_panel(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['Date'])
    monthly = (
        df.set_index('Date')[COLUMNS]
        .resample('ME')
        .last()
        .dropna(how='all')
    )
    complete = monthly.dropna(subset=COLUMNS).copy()
    complete.index.name = 'Date'
    return complete


def ns_loadings(tau_years: np.ndarray, lambda_value: float) -> np.ndarray:
    slope = (1.0 - np.exp(-lambda_value * tau_years)) / (lambda_value * tau_years)
    curvature = slope - np.exp(-lambda_value * tau_years)
    return np.column_stack([np.ones_like(tau_years), slope, curvature])


def ols_betas_for_panel(yields_df: pd.DataFrame, lambda_value: float):
    H = ns_loadings(TAU_YEARS, lambda_value)
    HtH_inv = np.linalg.inv(H.T @ H)
    betas = []
    residuals = []
    rmse = []
    for _, row in yields_df.iterrows():
        y = row[COLUMNS].to_numpy(dtype=float)
        beta = HtH_inv @ H.T @ y
        fitted = H @ beta
        err = y - fitted
        betas.append(beta)
        residuals.append(err)
        rmse.append(np.sqrt(np.mean(err ** 2)))
    betas = pd.DataFrame(betas, index=yields_df.index, columns=['level', 'slope', 'curvature'])
    residuals = pd.DataFrame(residuals, index=yields_df.index, columns=COLUMNS)
    return betas, residuals, float(np.mean(rmse))


def fit_var1(factors: pd.DataFrame):
    Y = factors.iloc[1:].to_numpy(dtype=float)
    Xlag = factors.iloc[:-1].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(Xlag)), Xlag])
    B = np.linalg.inv(X.T @ X) @ X.T @ Y
    c = B[0]
    A = B[1:].T
    resid = Y - X @ B
    Q = np.cov(resid.T, bias=False)
    return c, A, Q, resid


def kalman_filter_smoother(y_df, lambda_value, c, A, Q, R, x0=None, P0=None):
    H = ns_loadings(TAU_YEARS, lambda_value)
    Y = y_df[COLUMNS].to_numpy(dtype=float)
    nobs = Y.shape[0]
    nstate = A.shape[0]
    if x0 is None:
        x0 = np.linalg.solve(np.eye(nstate) - A, c)
    if P0 is None:
        P0 = solve_discrete_lyapunov(A, Q)

    x_pred = np.zeros((nobs, nstate))
    P_pred = np.zeros((nobs, nstate, nstate))
    x_filt = np.zeros((nobs, nstate))
    P_filt = np.zeros((nobs, nstate, nstate))

    x_prev = x0.copy()
    P_prev = P0.copy()
    for t in range(nobs):
        x_prior = c + A @ x_prev
        P_prior = A @ P_prev @ A.T + Q
        innovation = Y[t] - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)
        x_post = x_prior + K @ innovation
        P_post = (np.eye(nstate) - K @ H) @ P_prior
        x_pred[t] = x_prior
        P_pred[t] = P_prior
        x_filt[t] = x_post
        P_filt[t] = P_post
        x_prev = x_post
        P_prev = P_post

    x_smooth = x_filt.copy()
    for t in range(nobs - 2, -1, -1):
        J = P_filt[t] @ A.T @ np.linalg.inv(P_pred[t + 1])
        x_smooth[t] = x_filt[t] + J @ (x_smooth[t + 1] - x_pred[t + 1])

    return pd.DataFrame(x_filt, index=y_df.index, columns=['level', 'slope', 'curvature']), pd.DataFrame(x_smooth, index=y_df.index, columns=['level', 'slope', 'curvature'])


def reconstruct_curve(months, beta_row, lambda_value):
    H = ns_loadings(np.asarray(months, dtype=float) / 12.0, lambda_value)
    return H @ np.asarray(beta_row, dtype=float)


def forward_curve_from_spot(months, spot_rates):
    months = np.asarray(months, dtype=float)
    spot = np.asarray(spot_rates, dtype=float) / 100.0
    forwards, f_months = [], []
    for i in range(1, len(months)):
        t1 = months[i - 1] / 12.0
        t2 = months[i] / 12.0
        z1 = spot[i - 1]
        z2 = spot[i]
        f = ((1.0 + z2) ** t2 / (1.0 + z1) ** t1) ** (1.0 / (t2 - t1)) - 1.0
        forwards.append(f * 100.0)
        f_months.append(months[i])
    return np.asarray(f_months), np.asarray(forwards)


## 2. Modelo base: Diebold-Li + Kalman

In [ ]:
panel = load_monthly_panel(DATA_PATH)
grid = np.linspace(0.02, 0.20, 120)
scores = []
for lam in grid:
    _, _, avg_rmse = ols_betas_for_panel(panel, lam)
    scores.append((lam, avg_rmse))

lambda_df = pd.DataFrame(scores, columns=['lambda', 'avg_rmse'])
best_lambda = float(lambda_df.sort_values('avg_rmse').iloc[0]['lambda'])

ols_betas, cross_residuals, avg_rmse = ols_betas_for_panel(panel, best_lambda)
state_intercept, transition_matrix, state_cov, var_resid = fit_var1(ols_betas)
measurement_cov = np.diag(np.maximum(cross_residuals.var(axis=0, ddof=1).to_numpy(dtype=float), 1e-6))

filtered_betas, smoothed_betas = kalman_filter_smoother(panel, best_lambda, state_intercept, transition_matrix, state_cov, measurement_cov)

best_lambda, avg_rmse, panel.shape

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for i, factor in enumerate(['level', 'slope', 'curvature']):
    axes[i].plot(ols_betas.index, ols_betas[factor], label='OLS', alpha=0.55)
    axes[i].plot(filtered_betas.index, filtered_betas[factor], label='Filtered', alpha=0.8)
    axes[i].plot(smoothed_betas.index, smoothed_betas[factor], label='Smoothed', linewidth=2)
    axes[i].set_title(factor)
    axes[i].legend(loc='best')
plt.tight_layout()

## 3. Identificación de regímenes

Para identificar regímenes usamos una mezcla gaussiana de dos componentes sobre un conjunto de features derivados de los factores suavizados:
- `level`
- `slope`
- `curvature`
- volatilidad rolling de 6 meses sobre cambios en `level`

Eso permite capturar no solo el nivel de la curva, sino también si el sistema está operando en una etapa más estable o más volátil.

In [ ]:
regime_features = smoothed_betas.copy()
regime_features['d_level'] = regime_features['level'].diff()
regime_features['level_vol_6m'] = regime_features['d_level'].rolling(6).std()
regime_features = regime_features.drop(columns=['d_level']).dropna().copy()

scaler = StandardScaler()
X_regime = scaler.fit_transform(regime_features[['level', 'slope', 'curvature', 'level_vol_6m']])

gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
raw_labels = gmm.fit_predict(X_regime)
regime_probs = gmm.predict_proba(X_regime)

regime_frame = regime_features[['level', 'slope', 'curvature']].copy()
regime_frame['raw_regime'] = raw_labels

# ordenamos regímenes para que 0 = low-rate / 1 = high-rate según media de level
regime_means = regime_frame.groupby('raw_regime')['level'].mean().sort_values()
mapping = {old: new for new, old in enumerate(regime_means.index.tolist())}
regime_frame['regime'] = regime_frame['raw_regime'].map(mapping)

proba_cols = []
for raw in range(2):
    proba_cols.append((mapping[raw], regime_probs[:, raw]))
for final_regime, prob_series in sorted(proba_cols, key=lambda x: x[0]):
    regime_frame[f'prob_regime_{final_regime}'] = prob_series

regime_frame[['regime']].groupby('regime').size()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(smoothed_betas.index, smoothed_betas['level'], label='Smoothed level', color='tab:blue')
mask0 = regime_frame['regime'] == 0
mask1 = regime_frame['regime'] == 1
axes[0].scatter(regime_frame.index[mask0], regime_frame.loc[mask0, 'level'], s=20, label='Regime 0', alpha=0.7)
axes[0].scatter(regime_frame.index[mask1], regime_frame.loc[mask1, 'level'], s=20, label='Regime 1', alpha=0.7)
axes[0].set_title('Level suavizado con clasificación de regímenes')
axes[0].legend(loc='best')

axes[1].plot(regime_frame.index, regime_frame['prob_regime_0'], label='P(Regime 0)')
axes[1].plot(regime_frame.index, regime_frame['prob_regime_1'], label='P(Regime 1)')
axes[1].set_title('Probabilidades de régimen (Gaussian Mixture)')
axes[1].legend(loc='best')
plt.tight_layout()

## 4. Dinámica temporal por régimen

Una vez clasificado cada período, estimamos una ecuación `VAR(1)` distinta para cada régimen.

Esto nos permite comparar, por ejemplo:
- persistencia del `level` bajo cada estado;
- cambios en la interacción entre `slope` y `curvature`;
- varianzas de shock distintas según el régimen.

In [ ]:
regime_states = smoothed_betas.loc[regime_frame.index].copy()
regime_states['regime'] = regime_frame['regime']

def fit_var1_by_regime(states_df: pd.DataFrame, regime_value: int):
    sub = states_df.copy()
    Y = []
    Xlag = []
    for t in range(1, len(sub)):
        if sub['regime'].iloc[t] == regime_value and sub['regime'].iloc[t - 1] == regime_value:
            Y.append(sub[['level', 'slope', 'curvature']].iloc[t].to_numpy(dtype=float))
            Xlag.append(sub[['level', 'slope', 'curvature']].iloc[t - 1].to_numpy(dtype=float))
    Y = np.asarray(Y)
    Xlag = np.asarray(Xlag)
    if len(Y) < 10:
        raise ValueError(f'Not enough transitions for regime {regime_value}')
    X = np.column_stack([np.ones(len(Xlag)), Xlag])
    B = np.linalg.inv(X.T @ X) @ X.T @ Y
    c = B[0]
    A = B[1:].T
    resid = Y - X @ B
    Q = np.cov(resid.T, bias=False)
    return c, A, Q, resid, len(Y)


regime_models = {}
for r in [0, 1]:
    regime_models[r] = fit_var1_by_regime(regime_states, r)

transition_counts = np.zeros((2, 2), dtype=float)
labels = regime_frame['regime'].to_numpy(dtype=int)
for i in range(1, len(labels)):
    transition_counts[labels[i - 1], labels[i]] += 1
transition_probs = transition_counts / transition_counts.sum(axis=1, keepdims=True)

for r in [0, 1]:
    c_r, A_r, Q_r, _, nobs_r = regime_models[r]
    print('regime', r, 'obs', nobs_r)
    print('intercept', np.round(c_r, 4))
    print('A')
    print(np.round(A_r, 4))
    print()

print('transition probabilities')
pd.DataFrame(transition_probs, index=['from_0', 'from_1'], columns=['to_0', 'to_1'])

## 5. Forecast condicional al régimen actual

Tomamos el último estado suavizado y el último régimen identificado. A partir de ahí hacemos dos tipos de forecast:

1. **Forecast condicional fijo**: suponemos que el régimen actual persiste durante el horizonte.
2. **Forecast mixto por transición**: ponderamos el siguiente paso usando la matriz empírica de transición entre regímenes.

Esto no reemplaza un Markov-switching completo, pero sí permite ver cómo cambia la proyección cuando aceptamos que la dinámica no es única.

In [ ]:
def forecast_states_regime_fixed(last_state, c, A, steps):
    states = []
    x = np.asarray(last_state, dtype=float).copy()
    for _ in range(steps):
        x = c + A @ x
        states.append(x.copy())
    return np.asarray(states)


def forecast_states_mixture(last_state, regime_probs_now, regime_models, transition_probs, steps):
    x = np.asarray(last_state, dtype=float).copy()
    p = np.asarray(regime_probs_now, dtype=float).copy()
    states = []
    probs = []
    for _ in range(steps):
        next_p = p @ transition_probs
        forecasts = []
        for r in [0, 1]:
            c_r, A_r, _, _, _ = regime_models[r]
            forecasts.append(c_r + A_r @ x)
        x = next_p[0] * forecasts[0] + next_p[1] * forecasts[1]
        states.append(x.copy())
        probs.append(next_p.copy())
        p = next_p
    return np.asarray(states), np.asarray(probs)


last_date = regime_states.index[-1]
last_state = regime_states.loc[last_date, ['level', 'slope', 'curvature']].to_numpy(dtype=float)
last_regime = int(regime_states.loc[last_date, 'regime'])
last_probs = regime_frame.loc[last_date, ['prob_regime_0', 'prob_regime_1']].to_numpy(dtype=float)

horizons = [1, 3, 6, 12]
max_h = max(horizons)

fixed_forecast_states = forecast_states_regime_fixed(last_state, regime_models[last_regime][0], regime_models[last_regime][1], max_h)
mixture_forecast_states, future_regime_probs = forecast_states_mixture(last_state, last_probs, regime_models, transition_probs, max_h)

last_obs = panel.loc[last_date, COLUMNS].to_numpy(dtype=float)
current_curve = reconstruct_curve(TAU_MONTHS, last_state, best_lambda)
fixed_curves = {h: reconstruct_curve(TAU_MONTHS, fixed_forecast_states[h - 1], best_lambda) for h in horizons}
mix_curves = {h: reconstruct_curve(TAU_MONTHS, mixture_forecast_states[h - 1], best_lambda) for h in horizons}

last_date, last_regime, last_probs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(TAU_MONTHS, current_curve, linewidth=2.5, label=f'Current {last_date.date()}')
axes[0].scatter(TAU_MONTHS, last_obs, s=45, label='Observed')
for h in horizons:
    axes[0].plot(TAU_MONTHS, fixed_curves[h], marker='o', label=f'Fixed regime {h}M')
axes[0].set_title(f'Forecast spot con régimen fijo (current regime = {last_regime})')
axes[0].set_xlabel('Madurez (meses)')
axes[0].set_ylabel('Tasa')
axes[0].legend(ncol=2)

axes[1].plot(TAU_MONTHS, current_curve, linewidth=2.5, label=f'Current {last_date.date()}')
axes[1].scatter(TAU_MONTHS, last_obs, s=45, label='Observed')
for h in horizons:
    axes[1].plot(TAU_MONTHS, mix_curves[h], marker='o', label=f'Mixture {h}M')
axes[1].set_title('Forecast spot con mezcla de transiciones')
axes[1].set_xlabel('Madurez (meses)')
axes[1].set_ylabel('Tasa')
axes[1].legend(ncol=2)

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fm, ff = forward_curve_from_spot(TAU_MONTHS, current_curve)
axes[0].plot(fm, ff, linewidth=2.5, label='Current forward')
for h in horizons:
    fm_h, ff_h = forward_curve_from_spot(TAU_MONTHS, fixed_curves[h])
    axes[0].plot(fm_h, ff_h, marker='o', label=f'Fixed regime {h}M')
axes[0].set_title('Forward curves con régimen fijo')
axes[0].set_xlabel('Madurez forward (meses)')
axes[0].set_ylabel('Forward rate')
axes[0].legend(ncol=2)

axes[1].plot(fm, ff, linewidth=2.5, label='Current forward')
for h in horizons:
    fm_h, ff_h = forward_curve_from_spot(TAU_MONTHS, mix_curves[h])
    axes[1].plot(fm_h, ff_h, marker='o', label=f'Mixture {h}M')
axes[1].set_title('Forward curves con mezcla de regímenes')
axes[1].set_xlabel('Madurez forward (meses)')
axes[1].set_ylabel('Forward rate')
axes[1].legend(ncol=2)

plt.tight_layout()

In [ ]:
future_prob_df = pd.DataFrame(
    future_regime_probs[[h - 1 for h in horizons]],
    index=[f'{h}M' for h in horizons],
    columns=['P(regime 0)', 'P(regime 1)'],
)
future_prob_df

## 6. Lectura de resultados

Qué mirar en este notebook:

- si los regímenes identificados separan realmente períodos distintos de la curva;
- si los `VAR(1)` por régimen muestran persistencias distintas;
- si el forecast con mezcla se aparta materialmente del forecast de régimen fijo;
- si las curvas forward cambian de forma relevante bajo distintos regímenes.

Interpretación prudente:
esta extensión ya captura algo importante —que la dinámica de la curva no tiene por qué ser única—, pero todavía es una aproximación. El siguiente salto metodológico sería un modelo completo de Markov-switching state-space con estimación conjunta de estados y probabilidades de régimen.